# Homework 6 — Colab starter

Complete the assigned preparation before beginning.

## How to work in this notebook

This `.ipynb` is your **only working document**. The assignment PDF is a read-only copy of the same prompt. Do not edit or combine a `.qmd` file.

- Write reasoning in the designated text cells.
- Run or modify the starter code rather than pasting an unexplained replacement.
- Keep requested output, plots, excerpts, and raw AI evidence visible.
- Handwriting is welcome but never required. Typed Markdown/LaTeX is fully equivalent.
- If you insert a clear scan/photo, add one typed description or statistical conclusion for accessibility.

Before submission, restart and run all. Upload a PDF export and the completed `.ipynb` to the same Gradescope assignment. If image insertion fails, add one clearly labeled optional handwriting PDF; do not merge files.

In [ ]:
# Run this cell once. It downloads a small course helper file, then uses it
# to download this week's data and check that every file arrived intact.
from pathlib import Path
import urllib.request

HELPER_URL = "https://raw.githubusercontent.com/skgallagher/stat-methods-ai-public/main/course_helpers/course_setup.py"
if not Path("course_setup.py").exists():
    urllib.request.urlretrieve(HELPER_URL, "course_setup.py")

from course_setup import setup_course
data_dir = setup_course('camelyon17')


In [ ]:
# Libraries and a fixed random seed used in this notebook
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 2027
np.random.seed(SEED)  # makes random results repeatable


**Do not use AI for Problem 1.** Work the derivations yourself. AI is allowed in Problem 2 and required in Problem 3.

| | |
|---|---|
| **Out / due** | Tue Mar 23 / Tue Mar 30, 11:59pm |
| **Points** | 100: Problem 1 (45), Problem 2 (40), Problem 3 (15) |
| **Expected time** | About 5 hours after Labs 9 and 10. Contact course staff if setup or debugging alone takes more than 30 minutes. |
| **Reading** | Steegen, Tuerlinckx, Gelman, and Vanpaemel (2016), ["Increasing Transparency Through a Multiverse Analysis"](https://doi.org/10.1177/1745691616658637), *Perspectives on Psychological Science*, the introduction and the worked example. Optional, if you did not read it for Week 9: Recht, Roelofs, Schmidt, and Shankar (2019), ["Do ImageNet Classifiers Generalize to ImageNet?"](https://proceedings.mlr.press/v97/recht19a.html), Sections 1 and 2. |
| **Sources** | Mohanty, Hughes, and Salathé (2016), ["Using Deep Learning for Image-Based Plant Disease Detection"](https://doi.org/10.3389/fpls.2016.01419), *Frontiers in Plant Science*; Kish (1965), *Survey Sampling*, Wiley |
| **AI policy** | No AI for Problem 1. AI is allowed in Problem 2 and required in Problem 3. Submit the AI-use record at the end. |
| **Working file** | `hw06_starter.ipynb`. The PDF is a read-only copy of the prompt. |
| **Submit** | Notebook PDF and completed `.ipynb` in one Gradescope submission. |

## The situation

In Week 9 you reweighted a benchmark's accuracy to a client's mix of cases. In Week 10 you checked whether a decision changes when you make a different reasonable choice in the analysis. In this homework you do both. You reweight under several scenarios and see whether the decision changes. In Problem 1 the scenarios are different mixes of cases, and in Problem 2 they are different ways of grouping the cases. Then you compare the reweighted numbers with the client's own labels.

Problem 1 is a plant-disease app, worked by hand. Problem 2 returns to Lab 9's tumor model and a hospital the lab did not use, hospital 4. The starter reuses Lab 9's code for the patch grid, blue minus red, `accuracy_by_group()`, and `reweight()`. As in the Week 10 lab, you write down the alternatives and what would change the decision before you run them. In Problem 3 you ask an AI assistant how to check the analysis and sort its suggestions.

Problem 2 uses formulas from Problem 1, so do Problem 1 first.

In [ ]:
# Additional library for this homework
from statsmodels.stats.proportion import proportion_confint

# Read the patch table and the images, as in Lab 9. Row i of the table matches image i.
c17_dir = data_dir / "camelyon17"
patches = pd.read_csv(c17_dir / "patches.csv")
images = np.load(c17_dir / "patches.npz")["images"]

# correct = 1 if the model's call matches the pathologist's label, 0 if not.
patches["correct"] = (patches["predicted_tumor"] == patches["label"]).astype(int)

# Two measurements that need no label, each averaged over a patch's pixels:
# blue minus red (larger is bluer, as in Lab 9) and brightness (smaller is darker).
blue_minus_red = []
brightness = []
for i in range(len(images)):
    pixels = images[i].astype(float)
    blue_minus_red.append(np.mean(pixels[:, :, 2] - pixels[:, :, 0]))
    brightness.append(np.mean(pixels))
patches["blue_minus_red"] = blue_minus_red
patches["brightness"] = brightness

benchmark = patches[patches["group"] == "benchmark"]
hospital_4 = patches[patches["group"] == "hospital 4"]
print("Benchmark: ", len(benchmark), "patches from", benchmark["slide_id"].nunique(), "slides at hospitals 1-3")
print("Hospital 4:", len(hospital_4), "patches from", hospital_4["slide_id"].nunique(), "slides")
# Do not look at hospital 4's accuracy until Problem 2(d).

# Problem 1 - Reweighting a plant-disease app (45 points, no AI)

Show your work. A correct number without the steps behind it earns little credit. The setting is loosely based on Mohanty, Hughes, and Salathé (2016), whose plant-disease classifier, trained on lab photos of single leaves on plain backgrounds, was about 31% accurate on photos from other sources. The numbers below are made up. The effective sample size in (a)(iii) is from Kish (1965).

An app looks at a photo of a leaf and says whether the plant is diseased. Its benchmark has 1,000 labeled photos: 800 taken in a lab on a plain background, where the app is right on 768, and 200 taken in a field, where it is right on 160. A farmers' cooperative wants to use the app. Its members take most of their photos in the field on their phones, and a survey of members suggests that about 70% of their photos will be field photos, somewhere between 60% and 80%. Before seeing any results, the cooperative decided to adopt the app only if its accuracy on the cooperative's photos is at least 85%.

The notation is Lecture 9A's. For one photo, $C=1$ if the app is right and $C=0$ if not. $Z$ is where the photo was taken, lab or field, which anyone can see without a label. For each value $k$ of $Z$, $b_k$ and $c_k$ are the benchmark's and the cooperative's shares of photos (the two marginals of $Z$), and $p_k=P(C=1\mid Z=k)$ is the conditional accuracy. With only two kinds of photo, the benchmark's shares are $b_{\text{lab}}=0.8$ and $b_{\text{field}}=0.2$, and the cooperative's are $c_{\text{lab}}$ and $c_{\text{field}}=1-c_{\text{lab}}$.

a. **Marginal times conditional (12 points).**

   **(i) (3 points)** Starting from the law of total expectation, $E[C]=E\bigl[E[C\mid Z]\bigr]$, show that the accuracy is
   $$\theta=\sum_k P(Z=k)\,P(C=1\mid Z=k),$$
   so that the benchmark's accuracy is $\sum_k b_k\,p_k$ and the cooperative's is $\sum_k c_k\,p_k$, each with its own $p_k$. Which of the two pieces can the cooperative see in its own photos without labeling any, and which can it not?

   **(ii) (4 points)** Compute the app's accuracy on the benchmark, and the reweighted accuracy for the cooperative at $c_{\text{field}}=0.7$, so $c_{\text{lab}}=0.3$. State the assumption the reweighted number rests on, in terms of conditional accuracies.

   **(iii) (5 points)** At $c_{\text{field}}=0.7$, compute the standard error of the reweighted accuracy, treating the photos as independent,
   $$\mathrm{SE}=\sqrt{\sum_k c_k^2\,\frac{\hat p_k(1-\hat p_k)}{n_k}},$$
   and Kish's effective sample size, $n_{\text{eff}}=1\big/\sum_k c_k^2/n_k$. How many of the benchmark's 1,000 photos is the reweighted accuracy worth? Which group contributes more of the variance, and why?

### Your response - Problem 1(a)

Show your work. Type it in Markdown/LaTeX, or insert a clear photo of handwritten work.
If you insert a photo, add one typed sentence with your answer.

**(i)**  
TODO

**(ii)**  
TODO

**(iii)**  
TODO


b. **How much does $Z$ explain? (9 points).**

   **(i) (5 points)** Draw one photo at random from the benchmark's 1,000 photos. Its kind, $Z$, and whether the app is right on it, $C$, are random. The shares $b_k=P(Z=k)$, the conditional accuracies $p_k=E[C\mid Z=k]$, and the accuracy $\theta$ are fixed numbers. So $E[C\mid Z]$ is random only because $Z$ is. When $Z=\text{lab}$, which happens with probability $b_{\text{lab}}$, it equals $p_{\text{lab}}$. When $Z=\text{field}$, with probability $b_{\text{field}}$, it equals $p_{\text{field}}$. Write $\Delta=p_{\text{lab}}-p_{\text{field}}$ for the gap between the two conditional accuracies. Show that
   $$\operatorname{Var}\bigl(E[C\mid Z]\bigr)=b_{\text{lab}}\,b_{\text{field}}\,\Delta^2$$
   in four steps, then compute it.

   1. **The mean.** Write $E\bigl[E[C\mid Z]\bigr]$ as the sum over $k$ of $E[C\mid Z=k]\,P(Z=k)$, and use (a)(i) to explain why it equals $\theta$. (1 point)
   2. **The variance as an expected value.** Use $\operatorname{Var}(X)=E\bigl[(X-E[X])^2\bigr]$ to write $\operatorname{Var}\bigl(E[C\mid Z]\bigr)$ as the sum over $k$ of $\bigl(E[C\mid Z=k]-\theta\bigr)^2\,P(Z=k)$, and write out its two terms. (1 point)
   3. **The deviations.** Substitute $\theta$ from step 1 and show that $p_{\text{lab}}-\theta=b_{\text{field}}\,\Delta$ and $p_{\text{field}}-\theta=-b_{\text{lab}}\,\Delta$. Hint: $1-b_{\text{lab}}=b_{\text{field}}$. (1 point)
   4. **Factor.** Substitute step 3 into step 2, and use $b_{\text{lab}}+b_{\text{field}}=1$. (1 point)
   5. **The numbers.** Compute $\operatorname{Var}\bigl(E[C\mid Z]\bigr)$, and the share of $\operatorname{Var}(C)=\theta(1-\theta)$ on the benchmark that it makes up. (1 point)

   **(ii) (2 points)** Show that the reweighted accuracy minus the benchmark's accuracy equals $(c_{\text{field}}-b_{\text{field}})(p_{\text{field}}-p_{\text{lab}})$, and compute it at $c_{\text{field}}=0.7$.

   **(iii) (2 points)** In Lecture 9A, infrared explained under 1% of the variation in whether MegaDetector was right, and reweighting moved the estimate by 1.3 points. In 1 or 2 sentences, use (i) and (ii) to say why reweighting moves this estimate so much more.

### Your response - Problem 1(b)

Show your work. Type it in Markdown/LaTeX, or insert a clear photo of handwritten work.
If you insert a photo, add one typed sentence with your answer.

**(i) 1.**  
TODO

**(i) 2.**  
TODO

**(i) 3.**  
TODO

**(i) 4.**  
TODO

**(i) 5.**  
TODO

**(ii)**  
TODO

**(iii)**  
TODO


c. **Is the decision sensitive to the cooperative's mix? (14 points).** The survey gives only a range for the cooperative's share of field photos, so in this part treat it as unknown and write it as $c=c_{\text{field}}$.

   **(i) (4 points)** Write the reweighted accuracy $\theta(c)$ as a function of the field share $c$. Find the flip point $c^\ast$, the share at which $\theta(c^\ast)=0.85$, and compare it with the survey's 70%.

   **(ii) (5 points)** The cooperative's statistician wants a rule that accounts for uncertainty. It adopts the app only when the one-sided 95% lower bound is at least 0.85, that is, when
   $$\theta(c)-1.645\,\mathrm{SE}(c)\;\ge\;0.85,$$
   where $\mathrm{SE}(c)$ is the standard error from (a)(iii) with $c_{\text{field}}=c$ and $c_{\text{lab}}=1-c$,
   $$\mathrm{SE}(c)=\sqrt{(1-c)^2\,\frac{\hat p_{\text{lab}}(1-\hat p_{\text{lab}})}{n_{\text{lab}}}+c^2\,\frac{\hat p_{\text{field}}(1-\hat p_{\text{field}})}{n_{\text{field}}}}.$$
   Compute $\theta(c)$, $\mathrm{SE}(c)$, and the lower bound at $c=0.5$ and at $c=0.7$. What does this rule decide at each? The flip point for this rule is the share $c^\dagger$ at which $\theta(c^\dagger)-1.645\,\mathrm{SE}(c^\dagger)=0.85$. Without solving for it, explain why $0.5<c^\dagger<0.7$ and why $c^\dagger<c^\ast$.

   **(iii) (3 points)** Over the survey's range, 60% to 80% field photos, does either rule ever say to adopt? In 2 sentences, say whether the cooperative's decision is sensitive to its mix, and what it could do about not knowing $c$.

   **(iv) (2 points)** In 1 sentence each: why must the 85% threshold and the range of $c$ be set before the results are seen, and what would model shopping look like here?

### Your response - Problem 1(c)

Show your work. Type it in Markdown/LaTeX, or insert a clear photo of handwritten work.
If you insert a photo, add one typed sentence with your answer.

**(i)**  
TODO

**(ii)**  
TODO

**(iii)**  
TODO

**(iv)**  
TODO


d. **What conditioning misses (10 points).** The cooperative labels 200 of its own field photos. The app is right on 148 of them, or 74% (Wilson 95% interval 67.5% to 79.6%), against 80% on the benchmark's field photos.

   **(i) (2 points)** Which assumption from (a)(ii) does this result challenge?

   **(ii) (6 points)** Lecture 9A gave three ways that conditioning on $Z$ can miss a drop in accuracy.

   - **Something besides $Z$ differs.** Within one value of $Z$, the client's cases differ from the benchmark's in a way that affects whether the model is right.
   - **No overlap.** The client has cases that the benchmark has none of, so there is no conditional accuracy to borrow.
   - **The label means something different.** The same case would get one label from the client and another from the benchmark.

   Below are three possible reasons the app did worse on the cooperative's field photos. For each one, answer three questions. Which of the three ways is it? Could reweighting on lab versus field photos fix it? What data would show whether it is happening?

   1. The cooperative's field photos are taken on older phones in early-morning light, while the benchmark's field photos came from one research farm with good cameras at midday.
   2. Many members grow a crop the benchmark has no photos of.
   3. The cooperative's agronomists call a leaf diseased at the first small spot, while the benchmark's labelers required a clear lesion.

   **(iii) (2 points)** In (a)(ii), reweighting estimated the app's accuracy on the cooperative's photos at 84.8%. After seeing the 74% on the cooperative's own field photos, do you think the app's actual accuracy on the cooperative's photos is higher or lower than 84.8%, and why? What data would the cooperative need to estimate that accuracy directly, without relying on the benchmark? Answer in 1 or 2 sentences written for the cooperative.

### Your response - Problem 1(d)

Show your work. Type it in Markdown/LaTeX, or insert a clear photo of handwritten work.
If you insert a photo, add one typed sentence with your answer.

**(i)**  
TODO

**(ii) 1.**  
TODO

**(ii) 2.**  
TODO

**(ii) 3.**  
TODO

**(iii)**  
TODO


# Problem 2 - Does hospital 4's answer depend on how Z is defined? (40 points)

The patch data from Lab 9 include a hospital the lab did not use. Hospital 4 has 300 patches from 10 slides, and the model never saw any of its slides. Its pathology lead will use the model to flag patches only if the one-sided 95% lower bound for its accuracy on hospital 4's patches is at least 0.85. As in Lab 9, the patches are about half tumor, so every accuracy here describes this balanced mix.

You will reweight the benchmark's accuracy to hospital 4 in several ways, decide in advance what would change the decision, and only then look at hospital 4's labels. AI help with code is allowed. Record any substantial help in the AI-use table.

a. **Look first (5 points).** Run the code in Problem 2(a). It shows four patches without tumor and four with tumor from each of the four hospitals. It also draws histograms of blue minus red and of brightness for the benchmark and hospital 4. Brightness is a patch's average pixel value, so smaller is darker.

   1. Describe how hospital 4's patches differ from the benchmark's, in 2 or 3 sentences. (2 points)
   2. Report the share of hospital 4's patches outside the benchmark's range on each measure, and the share that falls in the benchmark's darkest third. (3 points)

In [ ]:
# Problem 2(a), step 1: four patches without tumor, then four with tumor, from each hospital (as in Lab 9)
fig, axes = plt.subplots(4, 8, figsize=(16, 9))
for row, hospital in enumerate([1, 2, 3, 4]):
    here = patches[patches["hospital"] == hospital]
    no_tumor = here[here["label"] == 0].sample(4, random_state=SEED)
    tumor = here[here["label"] == 1].sample(4, random_state=SEED)
    rows_here = pd.concat([no_tumor, tumor])
    for col in range(8):
        i = rows_here.index[col]
        axes[row, col].imshow(images[i])
        axes[row, col].set_title(f"hospital {hospital}, " + ("tumor" if patches.loc[i, "label"] == 1 else "no tumor"), fontsize=9)
        axes[row, col].axis("off")
plt.tight_layout()
plt.show()

# Problem 2(a), step 2: the two measurements, benchmark against hospital 4
# Gray and orange, so neither group's color looks like a stain color
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, column, axis_label in [(axes[0], "blue_minus_red", "blue minus red (larger is bluer)"),
                               (axes[1], "brightness", "brightness (smaller is darker)")]:
    bins = np.linspace(patches[column].min(), patches[column].max(), 40)   # the same bins for both groups
    ax.hist(benchmark[column], bins=bins, alpha=0.6, density=True, color="#8a8a85", label="benchmark (hospitals 1-3)")
    ax.hist(hospital_4[column], bins=bins, alpha=0.6, density=True, color="#d95926", label="hospital 4")
    ax.set_xlabel(axis_label)
    ax.legend()
plt.tight_layout()
plt.show()

# Problem 2(a), step 3: how much of hospital 4 lies outside the benchmark's range
for column in ["blue_minus_red", "brightness"]:
    below = (hospital_4[column] < benchmark[column].min()).mean()
    above = (hospital_4[column] > benchmark[column].max()).mean()
    print(f"{column}: benchmark mean {benchmark[column].mean():.1f}, hospital 4 mean {hospital_4[column].mean():.1f}")
    print(f"    hospital 4 patches below the benchmark's range: {below:.1%}, above it: {above:.1%}")

# The benchmark's darkest third is every patch below the benchmark's 1/3 point of brightness
darkest_third_cut = np.quantile(benchmark["brightness"], 1 / 3)
print(f"Hospital 4 patches in the benchmark's darkest third: {(hospital_4['brightness'] < darkest_third_cut).mean():.1%}")

### Your response - Problem 2(a)

**1. How hospital 4's patches differ from the benchmark's (2-3 sentences):**  
TODO

**2. Share outside the benchmark's range on each measure, and share in the benchmark's darkest third:**  
TODO

b. **Decide before running anything (8 points).** The *primary analysis* is the one you would report if you had to pick a single analysis in advance. Here it is Lab 9's: $Z$ is blue minus red, cut into three groups at the benchmark's 1/3 and 2/3 points. A sensitivity analysis asks whether the decision changes under other reasonable choices of $Z$.

   Part (a) suggests two kinds of choice to vary. The first is *which measurement* to group by: hospital 4 differs from the benchmark mostly in brightness, not in color. The second is *how many groups* to cut it into. Nothing in the data says three is the right number. More groups make the patches within a group more alike, so the conditional accuracy is more likely to carry over, but each group holds fewer benchmark patches, so the effective sample size falls (Lecture 9A). Five groups is one finer choice.

   Here are the primary analysis and four possible alternatives:

   0. **Primary:** blue minus red, cut at the benchmark's thirds (3 groups).
   1. Blue minus red, cut at the benchmark's fifths (5 groups).
   2. Brightness, cut at the benchmark's thirds (3 groups).
   3. Brightness, cut at the benchmark's fifths (5 groups).
   4. Blue minus red in thirds and brightness in thirds at the same time (9 groups). Each patch's group depends on both its color group from 0 and its brightness group from 2.

   Before you run part (c), fill in the notebook's table for alternatives 1 to 4. For each one, say what it changes about $Z$ compared with the primary analysis, and why it is a reasonable choice. Then say whether you expect its reweighted accuracy to be higher or lower than the primary analysis's estimate (or "unknown," with a reason), and whether you expect it to change the decision. Finally, name the one alternative most likely to change the decision, and say why. This part is graded on its reasoning, not on whether your predictions come true.

### Your response - Problem 2(b)

Fill in this table and save the notebook **before** you run part (c).

| Alternative | What it changes about Z, compared with the primary | Why it is a reasonable choice | Higher or lower than the primary's estimate? | Will the decision change? |
|---|---|---|---|---|
| 1. color fifths | TODO | TODO | TODO | TODO |
| 2. brightness thirds | TODO | TODO | TODO | TODO |
| 3. brightness fifths | TODO | TODO | TODO | TODO |
| 4. color thirds and brightness thirds (9 groups) | TODO | TODO | TODO | TODO |

**The alternative most likely to change the decision, and why:**  
TODO

c. **The sensitivity table (10 points).** Run the code in Problem 2(c). It assigns every patch to a group under each of the five specifications in (b), reweights the benchmark's accuracy to hospital 4 for each one, and prints the results as a table. Part 1 asks you to finish the code first.

   1. **Fill in the groups.** The function `groups_for()` has one case for each of the five specifications, and each case calls `cut_into_groups(column, n_groups, data)`. Replace each `TODO` with the column, `"blue_minus_red"` or `"brightness"`, and the number of groups, 3 or 5, for that specification. Alternative 4 needs two calls, one for color and one for brightness. The cell stops with a message until every argument is filled in. (2 points)
   2. **Explain the code.** In 3 or 4 sentences, explain what `cut_into_groups()` does, and what the loop over `specifications` computes for each specification. Say which numbers use the benchmark's labels and which use only hospital 4's patches, without their labels. (2 points)
   3. **Complete the table.** Fill in the `lower_bound` and `decision` columns. The other columns are already computed: the number of groups, the reweighted estimate, its standard error, and the effective sample size. (2 points)
   4. Look at the figure of the five estimates with their 95% intervals and the 0.85 line. Which specification, if any, changes the decision? (1 point)
   5. Explain why the effective sample size falls as $Z$ gets finer. Then use the table of alternative 4's nine groups to find the group that holds the most hospital 4 patches, and report how many benchmark patches are in it. (2 points)
   6. Compare your predictions in (b) with the table. Which came true, and which did not? (1 point)

In [ ]:
# Problem 2(c): reweight the benchmark's accuracy to hospital 4 in five ways

# Lab 9's two functions, unchanged
def accuracy_by_group(groups, correct):
    """One row per group: cases, the benchmark's share, and accuracy with a Wilson interval."""
    table = pd.DataFrame({"group": groups, "correct": correct})
    total = len(table)
    rows = []
    for group_name, cases in table.groupby("group"):
        n = len(cases)                       # cases in this group
        right = int(cases["correct"].sum())  # cases the model got right
        low, high = proportion_confint(right, n, method="wilson")
        rows.append({"group": group_name, "cases": n, "benchmark_share": n / total,
                     "accuracy": right / n, "ci_low": low, "ci_high": high})
    return pd.DataFrame(rows).sort_values("accuracy").reset_index(drop=True)


def reweight(by_group, shares):
    """Accuracy for a mix of groups, its standard error, and Kish's effective sample size."""
    total_share = sum(shares.values())
    estimate = 0.0
    variance = 0.0
    weight_sum = 0.0
    for _, row in by_group.iterrows():
        share = shares.get(row["group"], 0.0) / total_share  # shares are rescaled to add to 1
        p = row["accuracy"]
        n = row["cases"]
        estimate = estimate + share * p                   # share times accuracy, added up as in the table
        variance = variance + share**2 * p * (1 - p) / n  # each group's variance, times its share squared
        weight_sum = weight_sum + share**2 / n            # used for Kish's effective sample size
    return estimate, np.sqrt(variance), 1 / weight_sum


# Part 1: you fill in the arguments marked TODO in groups_for() below.
TODO = None


def cut_into_groups(column, n_groups, data):
    """Cut a measurement at the benchmark's quantiles and name each patch's group."""
    if column is None or n_groups is None:
        raise ValueError("Fill in the column and the number of groups in every call in groups_for(), then run this cell again.")
    cut_points = np.quantile(benchmark[column], np.linspace(0, 1, n_groups + 1)[1:-1])
    numbers = np.digitize(data[column], cut_points) + 1   # 1 is the lowest group
    names = []
    for number in numbers:
        names.append(f"{column} {number} of {n_groups}")
    return np.array(names)


def groups_for(specification, data):
    """The group of every patch in data under one specification of Z.

    Part 1: replace each TODO with the column ("blue_minus_red" or "brightness")
    and the number of groups (3 or 5) for that specification.
    """
    if specification == "0. primary: color thirds":
        return cut_into_groups(TODO, TODO, data)
    if specification == "1. color fifths":
        return cut_into_groups(TODO, TODO, data)
    if specification == "2. brightness thirds":
        return cut_into_groups(TODO, TODO, data)
    if specification == "3. brightness fifths":
        return cut_into_groups(TODO, TODO, data)
    if specification == "4. color thirds and brightness thirds":
        color = cut_into_groups(TODO, TODO, data)   # the color group, as in 0
        dark = cut_into_groups(TODO, TODO, data)    # the brightness group, as in 2
        names = []
        for i in range(len(color)):
            names.append(color[i] + " / " + dark[i])
        return np.array(names)


def hospital_4_shares(groups):
    """Hospital 4's share of patches in each group. These need no labels."""
    shares = {}
    for name in np.unique(groups):
        shares[name] = np.mean(groups == name)
    return shares


specifications = ["0. primary: color thirds", "1. color fifths", "2. brightness thirds",
                  "3. brightness fifths", "4. color thirds and brightness thirds"]

rows = []
for specification in specifications:
    benchmark_groups = groups_for(specification, benchmark)
    by_group = accuracy_by_group(benchmark_groups, benchmark["correct"].to_numpy())
    shares = hospital_4_shares(groups_for(specification, hospital_4))

    # Overlap check: a hospital 4 group with no benchmark patches cannot be reweighted.
    for name in shares:
        if name not in list(by_group["group"]):
            print("No benchmark patches in", name, "- hospital 4 share", round(shares[name], 3))

    estimate, se, n_eff = reweight(by_group, shares)
    rows.append({"specification": specification, "groups": len(by_group),
                 "estimate": estimate, "se": se, "n_eff": n_eff})

table = pd.DataFrame(rows)

# Part 3 (TODO): the one-sided 95% lower bound, estimate - 1.645 * se
table["lower_bound"] = np.nan
# Part 3 (TODO): the decision, "use" or "do not use", under the rule: use the model if the lower bound is at least 0.85
table["decision"] = "TODO"
display(table.round(3))

# Part 4: the five estimates with 95% intervals, in the order listed
fig, ax = plt.subplots(figsize=(8, 3.5))
y = np.arange(len(table))[::-1]   # the primary analysis at the top
ax.errorbar(table["estimate"], y, xerr=1.96 * table["se"], fmt="o", color="#155a8a", capsize=0)
ax.axvline(0.85, color="#c41230", linestyle="--", label="0.85, the pathology lead's line")
ax.set_yticks(y)
ax.set_yticklabels(table["specification"])
ax.set_xlabel("reweighted accuracy for hospital 4 (95% interval)")
ax.legend(loc="upper right")
plt.tight_layout()
plt.show()

# Part 5: alternative 4's nine groups, one row per group
nine_groups = "4. color thirds and brightness thirds"
cells = pd.DataFrame({"benchmark patches": pd.Series(groups_for(nine_groups, benchmark)).value_counts(),
                      "hospital 4 patches": pd.Series(groups_for(nine_groups, hospital_4)).value_counts()})
display(cells.fillna(0).astype(int).sort_values("hospital 4 patches", ascending=False))

### Your response - Problem 2(c)

**1.** Fill in the arguments marked `TODO` in `groups_for()` in the code cell above.

**2. What `cut_into_groups()` does, what the loop computes for each specification, and which numbers use labels (3-4 sentences):**  
TODO

**3.** Fill in the `lower_bound` and `decision` columns in the code cell above.

**4. Which specification, if any, changes the decision:**  
TODO

**5. Why the effective sample size falls as Z gets finer, and the group with the most hospital 4 patches:**  
TODO

**6. Which of your predictions in (b) came true, and which did not:**  
TODO

d. **Now look at hospital 4's labels (9 points).** Run the code in Problem 2(d). It has three blocks, one for each of questions 1 to 3.

   1. The first block reports hospital 4's accuracy on its 300 labeled patches and its difference from the benchmark's accuracy. The 95% interval for the difference comes from `slide_bootstrap_difference()`, the function from Lab 9. It resamples whole slides at both places, keeping each slide's patches together, because patches from the same slide are related. Report hospital 4's accuracy, the difference, and its interval. Did any specification in (c) predict an accuracy this low? (2 points)
   2. The second block compares the benchmark's and hospital 4's accuracy within each brightness third. Which assumption failed, and how does the table show it? (2 points)
   3. The standard errors in (c) treat every patch as independent, which Lab 9 showed is not true. The third block, `slide_bootstrap_reweighted()`, redoes the primary analysis 2,000 times. Each time, it draws the benchmark's 9 slides and hospital 4's 10 slides with replacement, keeping each slide's patches together, and reweights again. The 5th percentile of the 2,000 estimates is a one-sided 95% lower bound that accounts for the slides. Report it. Would this lower bound have warned hospital 4 that the model's accuracy could be below 0.85? (2 points)
   4. A teammate notices that brightness in fifths gave the estimate closest to hospital 4's labels and wants to report that specification. In 2 sentences, explain what is wrong with this, and why it could not be done for a hospital that has no labels yet. (3 points)

In [ ]:
# Problem 2(d), part 1: hospital 4's own labels
print(f"Hospital 4 accuracy: {hospital_4['correct'].mean():.3f} ({hospital_4['correct'].sum()} of {len(hospital_4)} patches)")
print(f"Benchmark accuracy:  {benchmark['correct'].mean():.3f}")


# Lab 9's slide bootstrap for a difference in accuracy, unchanged
def slide_bootstrap_difference(new, old, B=4000, seed=SEED):
    """Percentile interval for accuracy(new) - accuracy(old), resampling whole slides in each."""
    rng = np.random.default_rng(seed)
    # Keep each slide's patches together: one array of 0s and 1s per slide
    new_slides = []
    for _, cases in new.groupby("slide_id"):
        new_slides.append(cases["correct"].to_numpy())
    old_slides = []
    for _, cases in old.groupby("slide_id"):
        old_slides.append(cases["correct"].to_numpy())
    differences = []
    for b in range(B):
        # Draw as many slides as each place has, with replacement
        new_pick = rng.integers(0, len(new_slides), len(new_slides))
        old_pick = rng.integers(0, len(old_slides), len(old_slides))
        # Pool the patches on the drawn slides and compute each accuracy
        new_accuracy = np.concatenate([new_slides[i] for i in new_pick]).mean()
        old_accuracy = np.concatenate([old_slides[i] for i in old_pick]).mean()
        differences.append(new_accuracy - old_accuracy)
    # The middle 95% of the 4,000 differences
    return np.percentile(differences, [2.5, 97.5])


low, high = slide_bootstrap_difference(hospital_4, benchmark)
print(f"Hospital 4 minus benchmark: {hospital_4['correct'].mean() - benchmark['correct'].mean():+.3f} "
      f"(95% interval {low:+.3f} to {high:+.3f}, resampling slides)")

# Problem 2(d), part 2: accuracy within each brightness third
benchmark_dark = cut_into_groups("brightness", 3, benchmark)
hospital_4_dark = cut_into_groups("brightness", 3, hospital_4)
rows = []
for name in np.unique(benchmark_dark):
    in_benchmark = benchmark_dark == name
    in_hospital_4 = hospital_4_dark == name
    rows.append({"brightness group": name,
                 "benchmark patches": in_benchmark.sum(),
                 "benchmark accuracy": benchmark["correct"].to_numpy()[in_benchmark].mean(),
                 "hospital 4 patches": in_hospital_4.sum(),
                 "hospital 4 accuracy": hospital_4["correct"].to_numpy()[in_hospital_4].mean()})
display(pd.DataFrame(rows).round(3))


# Problem 2(d), part 3: resample whole slides for the primary analysis
def slide_bootstrap_reweighted(specification, B=2000, seed=SEED):
    """B reweighted estimates, each from benchmark slides and hospital 4 slides drawn with replacement."""
    rng = np.random.default_rng(seed)
    benchmark_slides = []
    for _, cases in benchmark.groupby("slide_id"):
        benchmark_slides.append(cases)
    hospital_4_slides = []
    for _, cases in hospital_4.groupby("slide_id"):
        hospital_4_slides.append(cases)

    estimates = []
    for b in range(B):
        # Draw as many slides as each place has, with replacement
        benchmark_pick = rng.integers(0, len(benchmark_slides), len(benchmark_slides))
        hospital_4_pick = rng.integers(0, len(hospital_4_slides), len(hospital_4_slides))
        benchmark_sample = pd.concat([benchmark_slides[i] for i in benchmark_pick])
        hospital_4_sample = pd.concat([hospital_4_slides[i] for i in hospital_4_pick])

        # Redo the reweighting on the resampled patches, with the same cut points
        by_group = accuracy_by_group(groups_for(specification, benchmark_sample),
                                     benchmark_sample["correct"].to_numpy())
        shares = hospital_4_shares(groups_for(specification, hospital_4_sample))
        estimate, se, n_eff = reweight(by_group, shares)
        estimates.append(estimate)
    return np.array(estimates)


reweighted_estimates = slide_bootstrap_reweighted("0. primary: color thirds")   # takes a few seconds
print("Primary analysis, resampling slides")
print("  95% interval:", np.round(np.quantile(reweighted_estimates, [0.025, 0.975]), 3))
print("  one-sided 95% lower bound (the 5th percentile):", round(np.quantile(reweighted_estimates, 0.05), 3))

### Your response - Problem 2(d)

**1. Hospital 4's accuracy, its difference from the benchmark with the slide interval, and whether any specification predicted it:**  
TODO

**2. Which assumption failed, and how the brightness table shows it:**  
TODO

**3. The slide-resampled lower bound, and whether it would have warned hospital 4:**  
TODO

**4. What is wrong with reporting the specification closest to the labels (2 sentences):**  
TODO

e. **A paragraph for the pathology lead (8 points).** Write one paragraph of 3 to 5 sentences for hospital 4's pathology lead. You do not need a sentence for each item below, but the paragraph should contain all of this information.

   - The primary analysis's reweighted accuracy and its lower bound.
   - What alternatives 1 to 4 changed (the measurement and the number of groups), and why those changes were reasonable.
   - How far the estimate moved across the five analyses, and whether the decision changed.
   - The assumption that none of the five analyses could check, and what hospital 4's own labels showed about it.
   - What labeled data hospital 4 should collect before relying on the model, and why.

### Your response - Problem 2(e)

**A paragraph for hospital 4's pathology lead (3-5 sentences, with every item in the list):**  
TODO

# Problem 3 - Sensitivity analysis or model shopping? (15 points)

a. **Ask (5 points).** Paste this prompt into an AI assistant, and preserve your prompt and its complete response.

   > I'm evaluating a model that labels 96-by-96-pixel patches of lymph-node tissue slides as tumor or not tumor. It uses ImageNet ResNet-18 features with a logistic regression on top. On a benchmark of 900 labeled patches from 9 slides at three hospitals, it is right on 90.7% of patches. A new hospital has 300 patches from 10 slides, and I have not used their labels. To estimate the model's accuracy at the new hospital, I cut the patches into three groups by stain color (the average of blue minus red over a patch's pixels), computed the benchmark's accuracy in each group, and averaged those accuracies using the new hospital's share of patches in each group. That gave 90.4%, with a one-sided 95% lower bound of 88.7%. The hospital will use the model only if the lower bound is at least 85%. List three ways to check whether this decision depends on the analytic choices I made.

   Then sort the assistant's three suggestions in the notebook's table. A suggestion is a *sensitivity analysis* if it varies a defensible choice about the data, the groups, the population, or the decision rule, made for a reason before the result is known. It is *model shopping* if it mainly tries to make the number better or closer to a target. Some suggestions can be either, depending on when and why the choice is made, so say which when that happens. Give a statistical reason for each.

### Your response - Problem 3(a)

**My prompt:**  
TODO

**The assistant's complete response:**  
TODO

| Suggestion | Sensitivity analysis, model shopping, or either | Statistical reason |
|---|---|---|
| 1. TODO | TODO | TODO |
| 2. TODO | TODO | TODO |
| 3. TODO | TODO | TODO |

b. **Check (5 points).** Pick the suggestion that is most misleading, either because it could only be done after seeing the new hospital's labels or because it cannot test the assumption that failed in Problem 2(d). In 2 or 3 sentences, explain the problem. If you think none of the three is misleading, say so instead, explain why each one is a fair check, and say whether any of them could have caught the failure in Problem 2(d).

### Your response - Problem 3(b)

**The most misleading suggestion and the problem with it, or why none is misleading (2-3 sentences):**  
TODO

c. **Plan for the next hospital (3 points).** Suppose a third hospital asks to use the model, and you have not seen its labels. Using the assistant's suggestions and what you found in Problem 2, write a short plan that you would commit to before running anything. Give the primary analysis you would report, one alternative and why it is reasonable, and the rule that turns the results into a decision, for example "use the model only if both lower bounds are at least 0.85." Then say what you will report if the alternative gives a different decision from the primary analysis.

### Your response - Problem 3(c)

**Primary analysis:**  
TODO

**One alternative, and why it is reasonable:**  
TODO

**The rule that turns the results into a decision:**  
TODO

**What I will report if the alternative gives a different decision:**  
TODO

## Required AI-use record (2 points)

Record only the **initial prompt** for each use. Do not paste follow-up prompts or the full conversation into this table.

| Assignment part | Tool | Purpose | Initial prompt only | What I checked | What changed after checking | Decision I made myself |
|---|---|---|---|---|---|---|
| Problem 2 (if AI was used) | TODO | TODO | TODO | TODO | TODO | TODO |
| Problem 3 | TODO | TODO | TODO | TODO | TODO | TODO |

## Final submission check

- [ ] I restarted the runtime and ran all cells from top to bottom.
- [ ] Every requested denominator, table, figure, excerpt, and interpretation is visible.
- [ ] Any handwritten images are legible and each has a typed description or statistical conclusion.
- [ ] Required raw AI prompts/outputs and the AI-use record are preserved.
- [ ] I opened my downloaded PDF and `.ipynb` before uploading them.

The PDF is the primary grading surface; the notebook is the executable record. Both represent the same work.